# 02 — Player Selection

**Purpose:** Joins verified player names onto every stroke, counts strokes per player across ALL matches, applies the ≥200 stroke threshold, and documents which players are included or excluded and why.

**Outputs:**
- `EDA/data/shuttleset_filtered_players.parquet` — final analysis ready dataset
- `EDA/data/included_players.csv` — players that meet threshold of ≥ 200 strokes worth of data
- `EDA/data/excluded_players.csv` — players cut, with documented reasons

In [1]:
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("__file__").resolve().parent))
from config import OUTPUT_DIR, MIN_STROKES

In [2]:
RALLY_KEY = ["match_id", "set_num", "rally"]

In [3]:
# Load Data

def load():
    clean_path = OUTPUT_DIR / "shuttleset_clean.parquet"
    map_path   = OUTPUT_DIR / "player_map.csv"

    for p in [clean_path, map_path]:
        if not p.exists():
            print(f"ERROR: {p.name} not found.")
            print("Run step00 and step01 first.")
            sys.exit(1)

    df= pd.read_parquet(clean_path)
    player_map = pd.read_csv(map_path)

    print(f"Strokes loaded : {len(df):,}")
    print(f"Matches in map : {len(player_map)}")
    return df, player_map

In [4]:
# Join real player names onto every stroke

def join_player_names(df: pd.DataFrame, player_map: pd.DataFrame) -> pd.DataFrame:
    print("**JOINING PLAYER NAMES**")

    df = df.merge(player_map, on="match_id", how="left")

    df["player_name"]   = df.apply(lambda r: r["A"] if r["player"] == "A" else r["B"], axis=1)
    df["opponent_name"] = df.apply(lambda r: r["B"] if r["player"] == "A" else r["A"], axis=1)
    df = df.drop(columns=["A", "B"])

    null_names = df["player_name"].isna().sum()
    if null_names > 0:
        print(f"{null_names} strokes have no player_name.")
        print("These match_ids are missing from player_map.csv:")
        missing_ids = df[df["player_name"].isna()]["match_id"].unique()
        for m in missing_ids:
            print(f"   - {m}")
    else:
        print(f"All {len(df):,} strokes have a player_name assigned")

    return df

In [5]:
# Count strokes per player

def count_strokes(df: pd.DataFrame) -> pd.DataFrame:
    print("**STROKE COUNTS PER PLAYER**")

    counts = (
        df.groupby("player_name")
        .agg(
            stroke_count=("stroke_id", "count"),
            match_count=("match_id",   "nunique"),
            rally_count=("rally",      "nunique"),
        )
        .reset_index()
        .sort_values("stroke_count", ascending=False)
        .reset_index(drop=True)
    )

    print(counts.to_string(index=False))
    return counts

In [6]:
# Apply threshold

def apply_threshold(counts: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    print(f"THRESHOLD: ≥ {MIN_STROKES} strokes")

    included = counts[counts["stroke_count"] >= MIN_STROKES].copy()
    excluded = counts[counts["stroke_count"] <  MIN_STROKES].copy()

    excluded["exclusion_reason"] = excluded["stroke_count"].apply(
        lambda n: f"only {n} strokes (threshold: {MIN_STROKES})"
    )

    print(f"\nINCLUDED — {len(included)} players:")
    print(included[["player_name", "stroke_count", "match_count"]].to_string(index=False))

    if len(excluded) > 0:
        print(f"\nEXCLUDED — {len(excluded)} players:")
        print(excluded[["player_name", "stroke_count", "exclusion_reason"]].to_string(index=False))
    else:
        print("\nNo players excluded — all meet the threshold")

    return included, excluded

In [7]:
# Save selected player data

def save_outputs(df: pd.DataFrame, included: pd.DataFrame, excluded: pd.DataFrame):
    included_names = set(included["player_name"])
    df_final = df[df["player_name"].isin(included_names)].copy()

    analysis_path  = OUTPUT_DIR / "shuttleset_filtered_players.parquet"
    included_path  = OUTPUT_DIR / "included_players.csv"
    excluded_path  = OUTPUT_DIR / "excluded_players.csv"

    df_final.to_parquet(analysis_path, index=False)
    included.to_csv(included_path, index=False)
    excluded.to_csv(excluded_path, index=False)

    print(f"\nSaved: {analysis_path}")
    print(f"Saved: {included_path}")
    print(f"Saved: {excluded_path}")

    return df_final

In [8]:
# Main

def main():
    print("**PLAYER SELECTION**")

    df, player_map = load()
    df= join_player_names(df, player_map)
    counts= count_strokes(df)
    included, excluded = apply_threshold(counts)
    df_final= save_outputs(df, included, excluded)

    print(f"  Players   : {df_final['player_name'].nunique()}")
    print(f"  Strokes   : {len(df_final):,}")
    print(f"  Matches   : {df_final['match_id'].nunique()}")
    print(f"  Rallies   : {df_final.groupby(RALLY_KEY).ngroups:,}")
    print(f"  Shot types: {df_final['type_en'].nunique()}")

In [9]:
if __name__ == "__main__":
    main()

**PLAYER SELECTION**
Strokes loaded : 26,457
Matches in map : 44
**JOINING PLAYER NAMES**
All 26,457 strokes have a player_name assigned
**STROKE COUNTS PER PLAYER**
                     player_name  stroke_count  match_count  rally_count
                  Chou Tien Chen          4541           11           42
                    Kento Momota          3142            6           44
                  Viktor Axelsen          2992           11           44
                Ng Ka Long Angus          2093            7           40
                 Anders Antonsen          1933            6           42
                Jonatan Christie          1721            5           39
                  Carolina Marin          1283            5           40
        Anthony Sinisuka Ginting          1265            5           41
                       Chen Long          1251            3           42
                     An Se Young           996            4           42
Hans-Kristian Solberg Vittinghu